# SEQ BirdDex Dataset Qualitative Review

This notebook audits downloaded bird images before they are allowed into model-training splits.

The review gate checks:
- whether the file opens correctly
- whether a bird-like subject is detected
- whether the bird is large enough in frame
- whether the bird crop is sharp enough
- whether the bird crop has usable contrast and exposure
- whether the image contains scope/vignette/camera artefacts
- whether the claimed label is visually plausible enough for training

Outputs:
- `data/reports/dataset_quality_review.jsonl`
- `data/images/review/accepted/`
- `data/images/review/quarantine/`
- `data/images/review/rejected/`

This notebook does not train the classifier. It cleans the candidate image pool.

In [ ]:
from __future__ import annotations

import os
import cv2
import json
import math
import shutil
import sys
import numpy as np
import pandas as pd

from matplotlib.pyplot import plot as plt
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from typing import Any, Literal, Iterator
from PIL import Image, ImageOps

try:
    from ultralytics import YOLO
    ULTRALYTICS_AVAILABLE = True
except Exception as exc:
    ULTRALYTICS_AVAILABLE = False
    YOLO = None
    print(f"[info] Ultralytics Unavailable: {type(exc).__name__}: {exc}")

def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until the BIRDIDEX package and pyproject are found."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "birdidex").exists():
            return candidate
    raise RuntimeError("Could not locate the BIRDIDEX repo root from the current notebook path.")


REPO_ROOT = find_repo_root()
SRC_ROOT = REPO_ROOT / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from birdidex import paths as bx_paths
from birdidex import images as bx_images
from birdidex import taxonomy as bx_taxonomy

DATA_ROOT = bx_paths.data_dir()
IMAGES_ROOT = bx_paths.images_dir()
RAW_ROOT = IMAGES_ROOT / "raw"
PROCESSED_ROOT = IMAGES_ROOT / "processed"
REVIEW_ROOT = IMAGES_ROOT / "review"
REPORTS_ROOT = DATA_ROOT / "reports"

QUALITY_REPORT_PATH = REPORTS_ROOT / "dataset_quality_review.jsonl"
IMAGE_RECORDS_PATH = bx_images.image_records_path(IMAGES_ROOT)

for path in [REVIEW_ROOT, REPORTS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print(f"repo: {REPO_ROOT}")
print(f"images: {IMAGES_ROOT}")
print(f"records: {IMAGE_RECORDS_PATH}")
print(f"quality report: {QUALITY_REPORT_PATH}")


## 1. Review Configuration

Purpose: every bad image should have a machine-readable reason.

In [ ]:
Decision = Literal["accepted", "quarantine", "rejected"]

# Quality Configuration Parameters

@dataclass(frozen=True)
class QualityConfig:
    # The Detector Config
    detector_model_name: str = "___.pt"
    detector_conf: float = 0.25
    bird_class_name: str = "bird"

    # Subject Size
    min_bbox_area_frac_classifier: float = 0.06
    min_bbox_area_frac_detector: float = 0.02
    min_bbox_area_frac: float = 0.95

    # Crop Quality of Image
    min_laplacian_var: float = 60.0
    min_crop_contrast_std: float = 18.0
    max_underexposed_frac: float = 0.25
    max_overexposed_frac: float = 0.15

    # Artefacts
    max_dark_border_frac: float = 0.30

    # Multi-subject handling
    max_bird_detections_for_classifier: int = 1

    # Decision policy
    quarantine_instead_of_reject: bool = True

CFG = QualityConfig()
asdict(CFG)

# Quality Review Parameters

@dataclass
class ImageQualityReview:
    image_path: str
    claimed_label: str | None
    provider: str | None
    provider_observation_id: str | None
    decision: Decision
    reasons: list[str]

    width: int | None = None
    height: int | None = None

    detector_conf: float | None = None
    bird_detections: int = 0
    bbox_xyxy: list[float] | None = None
    bbox_area_frac: float | None = None

    crop_laplacian_var: float | None = None
    crop_contrast_std: float | None = None
    crop_underexposed_frac: float | None = None
    crop_overexposed_frac: float | None = None
    dark_border_frac: float | None = None

    notes: str | None = None


## 2. Load Image Data for Review

In [ ]:
# -------- Loading Image MetaData -------- #

# TODO:
# - Confirm actual field names in image_records.jsonl.
# - Find which field points to local image path.
# - Find which field stores claimed class/species label.
# - Find provider observation ID, URL, license, quality grade, and taxon ID if present.

def read_jsnol(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"The required path does not exist: {QUALITY_REPORT_PATH}")

    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                print(f"[warn] bad JSONL line {Line_no}: {exc}")
    return rows

records = read_jsnol(IMAGE_RECORDS_PATH)
print(f"loaded records: {len(records)}")

records_df = pd.DataFrame(records)
records_df.head()

# -------- Resolve Local Image paths -------- #

IMAGE_SUFFIXES = frozenset(
    suffix.lower()
    for suffix in {
        ".jpg", ".jpeg", ".png", ".webp",
        ".arw", ".arq", ".srf", ".sr2",
        ".nef", ".nrw", ".cr3", ".cr2", ".crw",
    }
)

RAW_SUFFIXES = {".arw", ".arq", ".srf", ".sr2", ".nef", ".nrw", ".cr3", ".cr2", ".crw"}
RASTER_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp"}

if path.suffix.lower() in RAW_SUFFIXES:
    # TODO: either skip, convert using rawpy, or use embedded JPEG preview.
    pass

IMAGE_SUFFIX_TUPLE = tuple(sorted(IMAGE_SUFFIXES))

def has_image_suffix(path_like: str | os.PathLike[str]) -> bool:
    """
    Fast case-insensitive image suffix check.

    This avoids lowercasing the entire path. Only the extension is lowercased.
    """
    name = os.fspath(path_like)
    _, ext = os.path.splitext(name)
    return ext.lower() in IMAGE_SUFFIXES


def iter_image_files_fast(roots: list[Path] | tuple[Path, ...]) -> Iterator[Path]:
    """
    Fast recursive image discovery using os.scandir.

    This avoids pathlib.rglob overhead and avoids following symlinks.
    It yields Path objects only for files that pass the suffix filter.
    """
    stack: list[str] = []

    for root in roots:
        if root.exists():
            stack.append(os.fspath(root))

    while stack:
        current = stack.pop()

        try:
            with os.scandir(current) as entries:
                for entry in entries:
                    try:
                        if entry.is_dir(follow_symlinks=False):
                            stack.append(entry.path)
                            continue

                        if not entry.is_file(follow_symlinks=False):
                            continue

                        if has_image_suffix(entry.name):
                            yield Path(entry.path)

                    except OSError:
                        # Permission issues, broken entries, deleted files during scan, etc.
                        continue

        except OSError:
            # Missing/inaccessible directory.
            continue

def resolve_image_path(record: dict[str, Any]) -> Path | None:
    """
    Resolve an image path from a metadata record.

    Tries known metadata keys.
    Handles:
    - absolute paths
    - repo-relative paths
    - data-root-relative paths
    - images-root-relative paths
    """
    candidate_keys = [
        "local_path",
        "image_path",
        "path",
        "processed_path",
        "raw_path",
        "file_path",
    ]

    for key in candidate_keys:
        value = record.get(key)
        if not value:
            continue

        path = Path(str(value))
        if not path.is_absolute():
            path = REPO_ROOT / path

        if path.exists() and path.suffix.lower() in IMAGE_SUFFIXES:
            return path

    return None


def discover_images_fallback(sort_paths: bool = True) -> list[Path]:
    """
    Discover local images when no image metadata records are available.

    Sorting is useful for reproducibility, but it costs time on very large datasets.
    Set sort_paths=False for max speed.
    """
    roots = (RAW_ROOT, PROCESSED_ROOT)
    found = list(iter_image_files_fast(roots))

    if sort_paths:
        found.sort(key=lambda p: os.fspath(p).lower())

    return found


image_paths: list[tuple[Path, dict[str, Any]]] = []

if records:
    for record in records:
        path = resolve_image_path(record)
        if path is not None:
            image_paths.append((path, record))

    # Optional fallback: useful if records exist but paths are stale/missing.
    if not image_paths:
        print("[warn] records were loaded, but no local paths resolved. Falling back to filesystem scan.")
        image_paths = [(path, {}) for path in discover_images_fallback(sort_paths=True)]

else:
    image_paths = [(path, {}) for path in discover_images_fallback(sort_paths=True)]


print(f"reviewable local images: {len(image_paths)}")
image_paths[:3]

# -------- File Integrity Check -------- #

# Catch any corrupted or NaN files before wasting the detectors time

def open_image_rgb(path: Path) -> Image.Image:
    with Image.open(path) as img:
        return img.convert("RGB")

def file_check(path: Path) -> tuple[bool, list[str], dict[str, Any]]:
    reasons = []
    meta = {}

    if not path.exists():
        return False, ["file_empty"], meta

    if path.stat().st_size ==0:
        return False, ["file_empty"], meta
    
    try:
        img = open_image_rgb(path)
        meta["width"], meta["height"] = img.size
    except Exception as exc:
        return False, [f"cannot_open:{type(exc).__name__}"], meta

    if meta["width"] < 128 or meta["height"] < 128:
        reasons.append("image_too_small")
    
    return len(reasons) == 0, reasons, meta


## 3. Quality Detector Configuration

In [ ]:
# -------- Detector Setup -------- #

# TODO:
# - Run this on your 3 bad Peregrine examples and 3 good Peregrine examples.
# - Check whether COCO YOLO detects the tiny/silhouette bird at all.
# - If it misses too many birds, use a bird-specific detector later.
# - If it detects tiny garbage too often, keep detector but tighten bbox thresholds.

def load_bird_detector(cfg: QualityConfig):
    if not ULTRALYTICS_AVAILABLE:
        raise RuntimeError("Ultralytics is not installed.")

    model = YOLO(cfg.detector_model_name)
    return model

detector = load_bird_detector(CFG) if ULTRALYTICS_AVAILABLE else None

def detect_birds(path: Path, detector, cfg: QualityConfig) -> list[dict[str, Any]]:
    if detector is None:
        return []
    
    results = detector.predict(
        source=str(path),
        conf=cfg.detector_conf,
        verbose=False,
    )

    if not results:
        return []

    result = results[0]
    names = result.names

    detections = []

    if result.boxes is None:
        return detections

    for box in result.boxes:
        class_id = int(box.cls.item())
        class_name = str(names.get(class_id, class_id)).lower()
        conf = float(box.conf.item())
        xyxy = box.xyxy.cpu().numpy().reshape(-1).tolist()

        if cls_name == cfg.bird_class_name:
            detections.append(
                {
                    "class_id": cls_id,
                    "class_name": cls_name,
                    "conf": conf,
                    "xyxy": xyxy,
                }
            )

    detections.sort(key=lambda d: d["conf"], reverse=True)
    return detections

In [ ]:
# -------- Crop and Quality Metrics -------- #

def pil_to_cv_rgb(img: Image.Image) -> np.ndarray:
    return np.array(img.convert("RGB"))

def rgb_to_gray(rgb: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

def clip_bbox_xyxy(xyxy: list[float], width: int, height: int) -> tuple[int, int, int, int]:
    x1, y1, x2, y2 = xyxy
    x1 = max(0, min(width - 1, int(round(x1))))
    y1 = max(0, min(height - 1, int(round(y1))))
    x2 = max(0, min(width, int(round(x2))))
    y2 = max(0, min(height, int(round(y2))))
    return x1, y1, x2, y2


def crop_from_bbox(img: Image.Image, xyxy: list[float], pad_frac: float = 0.08) -> Image.Image:
    width, height = img.size
    x1, y1, x2, y2 = clip_bbox_xyxy(xyxy, width, height)

    bw = x2 - x1
    bh = y2 - y1
    pad_x = int(round(bw * pad_frac))
    pad_y = int(round(bh * pad_frac))

    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(width, x2 + pad_x)
    y2 = min(height, y2 + pad_y)

    return img.crop((x1, y1, x2, y2))


def laplacian_var(gray: np.ndarray) -> float:
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def exposure_stats(gray: np.ndarray) -> tuple[float, float]:
    under = float(np.mean(gray < 10))
    over = float(np.mean(gray > 245))
    return under, over


def dark_border_fraction(gray: np.ndarray, border_fraction: float = 0.08) -> float:
    h, w = gray.shape
    border = int(min(h, w) * border_fraction)

    if border <= 0:
        return 0.0

    mask = np.zeros_like(gray, dtype=bool)
    mask[:border, :] = True
    mask[-border:, :] = True
    mask[:, :border] = True
    mask[:, -border:] = True

    return float(np.mean(gray[mask] < 30))


def bbox_area_fraction(xyxy: list[float], width: int, height: int) -> float:
    x1, y1, x2, y2 = clip_bbox_xyxy(xyxy, width, height)
    area = max(0, x2 - x1) * max(0, y2 - y1)
    return float(area / (width * height))

## 4. Per-Image Review

In [ ]:
# -------- Per-Image Review Function -------- #

def review_one_image(path: Path, record: dict[str, Any], detector, cfg: QualityConfig) -> ImageQualityReview:
    claimed_label = (
        record.get("label")
        or record.get("common_name")
        or record.get("species")
        or record.get("class_name")
    )

    provider = record.get("provider")
    provider_observation_id = (
        record.get("provider_observation_id")
        or record.get("observation_id")
        or record.get("id")
    )

    ok, file_reasons, file_meta = basic_file_check(path)
    if not ok:
        return ImageQualityReview(
            image_path=str(path.relative_to(REPO_ROOT)),
            claimed_label=claimed_label,
            provider=provider,
            provider_observation_id=str(provider_observation_id) if provider_observation_id else None,
            decision="rejected",
            reasons=file_reasons,
            width=file_meta.get("width"),
            height=file_meta.get("height"),
        )

    img = open_image_rgb(path)
    width, height = img.size
    full_gray = rgb_to_gray(pil_to_cv_rgb(img))
    dark_border = dark_border_fraction(full_gray)

    detections = detect_birds(path, detector, cfg)
    reasons = []

    if dark_border > cfg.max_dark_border_frac:
        reasons.append("dark_border_or_scope_artefact")

    if not detections:
        return ImageQualityReview(
            image_path=str(path.relative_to(REPO_ROOT)),
            claimed_label=claimed_label,
            provider=provider,
            provider_observation_id=str(provider_observation_id) if provider_observation_id else None,
            decision="quarantine",
            reasons=reasons + ["no_bird_detected"],
            width=width,
            height=height,
            bird_detections=0,
            dark_border_frac=dark_border,
        )

    if len(detections) > cfg.max_bird_detections_for_classifier:
        reasons.append("multiple_bird_detections")

    best = detections[0]
    xyxy = best["xyxy"]
    area_frac = bbox_area_fraction(xyxy, width, height)

    if area_frac < cfg.min_bbox_area_frac_classifier:
        reasons.append("subject_too_small")

    if area_frac > cfg.max_bbox_area_frac:
        reasons.append("bbox_implausibly_large")

    crop = crop_from_bbox(img, xyxy)
    crop_gray = rgb_to_gray(pil_to_cv_rgb(crop))

    lap_var = laplacian_var(crop_gray)
    contrast_std = float(crop_gray.std())
    under, over = exposure_stats(crop_gray)

    if lap_var < cfg.min_laplacian_var:
        reasons.append("crop_too_blurry_or_low_detail")

    if contrast_std < cfg.min_crop_contrast_std:
        reasons.append("low_crop_contrast")

    if under > cfg.max_underexposed_frac:
        reasons.append("subject_underexposed_or_silhouette")

    if over > cfg.max_overexposed_frac:
        reasons.append("subject_overexposed")

    if not reasons:
        decision: Decision = "accepted"
    elif cfg.quarantine_instead_of_reject:
        decision = "quarantine"
    else:
        decision = "rejected"

    return ImageQualityReview(
        image_path=str(path.relative_to(REPO_ROOT)),
        claimed_label=claimed_label,
        provider=provider,
        provider_observation_id=str(provider_observation_id) if provider_observation_id else None,
        decision=decision,
        reasons=reasons,
        width=width,
        height=height,
        detector_conf=float(best["conf"]),
        bird_detections=len(detections),
        bbox_xyxy=[float(x) for x in xyxy],
        bbox_area_frac=area_frac,
        crop_laplacian_var=lap_var,
        crop_contrast_std=contrast_std,
        crop_underexposed_frac=under,
        crop_overexposed_frac=over,
        dark_border_frac=dark_border,
    )

In [ ]:
# -------- Run Quality Sweep -------- #

MAX_REVIEW_IMAGES = int(os.getenv("BIRDIDEX_MAX_REVIEW_IMAGES", "200"))

review_rows: list[ImageQualityReview] = []

for idx, (path, record) in enumerate(image_paths[:MAX_REVIEW_IMAGES], start=1):
    try:
        review = review_one_image(path, record, detector, CFG)
    except Exception as exc:
        review = ImageQualityReview(
            image_path=str(path.relative_to(REPO_ROOT)),
            claimed_label=record.get("label") or record.get("common_name"),
            provider=record.get("provider"),
            provider_observation_id=str(record.get("id")) if record.get("id") else None,
            decision="quarantine",
            reasons=[f"review_exception:{type(exc).__name__}"],
            notes=str(exc),
        )

    review_rows.append(review)

    if idx % 25 == 0:
        print(f"reviewed {idx}/{min(MAX_REVIEW_IMAGES, len(image_paths))}")

review_df = pd.DataFrame([asdict(row) for row in review_rows])
review_df.head()

## 5. Summary and Review of Results

In [ ]:
# -------- Summarise Failure Modes -------- #

decision_counts = review_df["decision"].value_counts(dropna=False)
display(decision_counts)

reason_counts = Counter()

for reasons in review_df["reasons"]:
    if isinstance(reasons, list):
        reason_counts.update(reasons)

reason_df = pd.DataFrame(
    [{"reason": reason, "count": count} for reason, count in reason_counts.most_common()]
)

display(reason_df)

In [ ]:
# -------- Visual Review Panels -------- #

def show_review_examples(review_df: pd.DataFrame, decision: str, max_examples: int = 12):
    subset = review_df[review_df["decision"] == decision].head(max_examples)

    if subset.empty:
        print(f"No examples for decision={decision}")
        return

    cols = 4
    rows = math.ceil(len(subset) / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, subset.iterrows()):
        path = REPO_ROOT / row["image_path"]
        img = open_image_rgb(path)
        ax.imshow(img)
        ax.axis("off")
        title = f"{row['decision']}\n{', '.join(row['reasons'][:2])}"
        ax.set_title(title, fontsize=8)

        bbox = row.get("bbox_xyxy")
        if isinstance(bbox, list) and len(bbox) == 4:
            x1, y1, x2, y2 = bbox
            rect = plt.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                fill=False,
                linewidth=2,
            )
            ax.add_patch(rect)

    for ax in axes[len(subset):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


show_review_examples(review_df, "accepted")
show_review_examples(review_df, "quarantine")
show_review_examples(review_df, "rejected")

In [ ]:
# -------- Threshold Calibration Table -------- #

# TODO:
# - Look at the accepted/quarantine boundary.
# - If too many good birds are quarantined, lower min_bbox_area_frac or min_laplacian_var.
# - If too many bad birds pass, raise min_bbox_area_frac and min_crop_contrast_std.
# - Tune thresholds separately for:
#   1. detector training images
#   2. classifier training images
#   3. field inference abstention

metric_cols = [
    "bbox_area_frac",
    "crop_laplacian_var",
    "crop_contrast_std",
    "crop_underexposed_frac",
    "crop_overexposed_frac",
    "dark_border_frac",
]

display(review_df.groupby("decision")[metric_cols].describe())

In [ ]:
# -------- Write Review JSONL -------- #

def write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\n")


review_dicts = [asdict(row) for row in review_rows]
write_jsonl(QUALITY_REPORT_PATH, review_dicts)

print(f"wrote {len(review_dicts)} review rows to {QUALITY_REPORT_PATH.relative_to(REPO_ROOT)}")

## 5. Sort Dataset Directories

In [ ]:
# -------- Copy/Move Review Sets -------- #

COPY_REVIEW_IMAGES = False


def safe_copy_review_image(row: dict[str, Any]) -> None:
    src = REPO_ROOT / row["image_path"]

    claimed = row.get("claimed_label") or "unknown"
    claimed_folder = str(claimed).lower().replace(" ", "_").replace("/", "_")

    dst_dir = REVIEW_ROOT / row["decision"] / claimed_folder
    dst_dir.mkdir(parents=True, exist_ok=True)

    dst = dst_dir / src.name

    if not dst.exists():
        shutil.copy2(src, dst)


if COPY_REVIEW_IMAGES:
    for row in review_dicts:
        safe_copy_review_image(row)

    print(f"copied reviewed images to {REVIEW_ROOT.relative_to(REPO_ROOT)}")
else:
    print("COPY_REVIEW_IMAGES=False, no files copied.")

In [ ]:
# -------- Build Accepted-only Manifest -------- #

accepted_df = review_df[review_df["decision"] == "accepted"].copy()

accepted_manifest_path = REPORTS_ROOT / "accepted_training_manifest.csv"
accepted_df.to_csv(accepted_manifest_path, index=False)

print(f"accepted images: {len(accepted_df)}")
print(f"manifest: {accepted_manifest_path.relative_to(REPO_ROOT)}")

In [ ]:
# -------- Split Leakage Check -------- #

# TODO:
# - Group images by provider_observation_id where available.
# - Keep all images from one observation in the same split.
# - If provider_observation_id is missing, use perceptual hash or source URL group.
# - Build splits only from accepted_df.